# Head OV Eigenanalysis

Eigenvalue structure of OV circuits: effective rank, copying scores,
unembedding projection, and cross-head alignment.

## Why This Matters

OV circuit head eigenanalysis analyzes the value-output projection that determines what information flows through attention. Understanding what each head copies when it attends to a position reveals the head's computational role in the model's circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_ov_eigenanalysis import (
    ov_eigenspectrum, ov_copying_score,
    ov_unembed_projection, ov_cross_head_alignment,
    ov_eigenanalysis_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## OV Eigenspectrum

Singular value decomposition of the OV circuit (W_V @ W_O)
reveals its effective dimensionality.

In [ ]:
for layer in range(4):
    for head in range(4):
        result = ov_eigenspectrum(model, layer=layer, head=head)
        print(f"  L{layer}H{head}: rank={result['effective_rank']:.2f}, "
              f"top_sv={result['top_sv']:.4f}, conc={result['sv_concentration']:.3f}")

## Copying Score

Does the OV circuit approximately copy (OV ≈ identity) or transform?

In [ ]:
for layer in range(4):
    for head in range(4):
        result = ov_copying_score(model, layer=layer, head=head)
        flag = ' [COPYING]' if result['is_copying'] else ''
        print(f"  L{layer}H{head}: identity={result['identity_score']:.4f}{flag}")

## Unembed Projection

Which vocabulary items are most affected by each head's OV circuit?

In [ ]:
for layer in range(4):
    for head in range(2):  # Show first 2 heads per layer
        result = ov_unembed_projection(model, layer=layer, head=head, top_k=3)
        tokens_str = [(t, f'{e:.3f}') for t, e in result['top_affected_tokens']]
        print(f"  L{layer}H{head}: top tokens={tokens_str}")

## Cross-Head OV Alignment

How aligned are OV circuits between heads in the same layer?

In [ ]:
for layer in range(4):
    result = ov_cross_head_alignment(model, layer=layer)
    print(f"  Layer {layer}: {result['n_aligned']} aligned pairs")
    for p in result['pairs']:
        flag = ' [ALIGNED]' if p['is_aligned'] else ''
        print(f"    H{p['heads'][0]}-H{p['heads'][1]}: cos={p['cosine']:.4f}{flag}")

## OV Eigenanalysis Summary

In [ ]:
result = ov_eigenanalysis_summary(model)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: mean_rank={p['mean_rank']:.2f}, "
          f"mean_copy={p['mean_copy_score']:.4f}")